# commec DNA sequence screening (one-click)

**First time opening this notebook:** run the single cell below once (click its ▶, or **Runtime → Run all**). That loads the input form — no visible code, just the form.

Then fill it in and click **Run**: installing `commec`, downloading the databases, and running the screen all happen automatically, no other setup required.

(First run only) `commec` and its dependencies install into an isolated [micromamba](https://mamba.readthedocs.io/en/latest/user_guide/micromamba.html) environment; this does **not** restart the Colab runtime, since `commec` is only ever invoked as a command-line subprocess.

In [ ]:
#@title commec screening — input & run { display-mode: "form" }
import html as html_lib
import os
import re
import subprocess

import ipywidgets as widgets
from IPython.display import HTML, display

MAMBA_ROOT_PREFIX = "/content/micromamba"
MICROMAMBA_BIN = f"{MAMBA_ROOT_PREFIX}/bin/micromamba"
COMMEC_ENV = "commec-env"
DB_DIR_DRIVE = "/content/drive/MyDrive/commec-databases"
DB_DIR_LOCAL = "/content/commec-databases"
OUTPUT_BASE_DIR = "/content/commec-output"
INPUT_FASTA_PATH = "/content/input.fasta"

os.environ["MAMBA_ROOT_PREFIX"] = MAMBA_ROOT_PREFIX

LAST_OUTPUT_DIR = None
LAST_NAME = None


# ---------------------------------------------------------------------------
# Pipeline internals
# ---------------------------------------------------------------------------

def _safe_job_name(name):
    name = re.sub(r"[^A-Za-z0-9_-]+", "-", name.strip())
    return name.strip("-")


def _stream(cmd, log_out):
    log_out.append_stdout(f"$ {cmd}\n")
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        log_out.append_stdout(line)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (exit {proc.returncode}): {cmd}")


def _commec_installed():
    return os.path.isdir(os.path.join(MAMBA_ROOT_PREFIX, "envs", COMMEC_ENV))


def _ensure_commec_installed(log_out, set_status):
    if _commec_installed():
        set_status("commec already installed.")
        return
    set_status("Installing micromamba + commec (one-time, a few minutes)...")
    os.makedirs(MAMBA_ROOT_PREFIX, exist_ok=True)
    if not os.path.exists(MICROMAMBA_BIN):
        _stream(
            f"curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | "
            f"tar -xvj -C {MAMBA_ROOT_PREFIX} bin/micromamba",
            log_out,
        )
    _stream(
        f"{MICROMAMBA_BIN} create -y -n {COMMEC_ENV} -c conda-forge -c bioconda commec",
        log_out,
    )


def _commec(args, log_out):
    _stream(f"{MICROMAMBA_BIN} run -n {COMMEC_ENV} commec {args}", log_out)


def _ensure_databases(db_dir, log_out, set_status):
    set_status("Downloading/checking screening databases...")
    os.makedirs(db_dir, exist_ok=True)
    _commec(f'setup -d "{db_dir}"', log_out)


def show_results(output_dir, name, report_out):
    html_path = os.path.join(output_dir, f"{name}_summary.html")
    json_path = os.path.join(output_dir, f"{name}.output.json")

    with report_out:
        if not os.path.exists(html_path):
            print(f"No report found at {html_path}")
            return

        with open(html_path, encoding="utf-8") as f:
            report_html = f.read()

        # The report ships its own full <html>/<body> document (with its own
        # background/margin rules). Rendering that directly into the notebook's
        # DOM via display(HTML(...)) would apply those body-level styles to the
        # notebook page itself. Putting it in an iframe (via srcdoc) gives the
        # report its own document, so its styles stay fully contained.
        iframe_html = (
            '<iframe srcdoc="' + html_lib.escape(report_html) + '" '
            'style="width:100%; height:80vh; border:1px solid #ccc;" '
            'sandbox="allow-scripts allow-same-origin allow-popups"></iframe>'
        )
        display(HTML(iframe_html))

        html_button = widgets.Button(description="Download HTML report", icon="download")
        json_button = widgets.Button(
            description="Download JSON report", icon="download", disabled=not os.path.exists(json_path)
        )

        def _download_html(b):
            from google.colab import files
            files.download(html_path)

        def _download_json(b):
            from google.colab import files
            files.download(json_path)

        html_button.on_click(_download_html)
        json_button.on_click(_download_json)
        display(widgets.HBox([html_button, json_button]))


def run_commec_pipeline(job_name, fasta_text, use_drive, log_out, report_out, set_status):
    global LAST_OUTPUT_DIR, LAST_NAME

    safe_job_name = _safe_job_name(job_name)
    if not safe_job_name:
        set_status("Please enter a job name before running.")
        return

    fasta_text = (fasta_text or "").strip()
    if not fasta_text:
        set_status("Please paste a FASTA sequence or upload a file above before running.")
        return

    if use_drive:
        from google.colab import drive
        set_status("Mounting Google Drive (approve the access prompt if asked)...")
        drive.mount("/content/drive")
        db_dir = DB_DIR_DRIVE
    else:
        db_dir = DB_DIR_LOCAL

    output_dir = os.path.join(OUTPUT_BASE_DIR, safe_job_name)
    os.makedirs(output_dir, exist_ok=True)
    with open(INPUT_FASTA_PATH, "w") as f:
        f.write(fasta_text if fasta_text.endswith("\n") else fasta_text + "\n")

    _ensure_commec_installed(log_out, set_status)
    _ensure_databases(db_dir, log_out, set_status)

    set_status("Running commec screen (this can take a while)...")
    _commec(
        f'screen "{INPUT_FASTA_PATH}" -d "{db_dir}" -o "{output_dir}" -t 4 --force', log_out
    )

    name = os.path.splitext(os.path.basename(INPUT_FASTA_PATH))[0]
    LAST_OUTPUT_DIR, LAST_NAME = output_dir, name
    set_status("Done — see the report below.")
    show_results(output_dir, name, report_out)


# ---------------------------------------------------------------------------
# Input form
# ---------------------------------------------------------------------------

job_name_input = widgets.Text(
    value="",
    placeholder="e.g. my-first-screen",
    description="Job name:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="50%"),
)

fasta_input = widgets.Textarea(
    value="",
    placeholder="Paste FASTA content here, or upload a file below to fill this box...",
    layout=widgets.Layout(width="100%", height="200px"),
)

upload_button = widgets.FileUpload(
    accept=".fasta,.fa,.fna,.txt",
    multiple=False,
    description="Upload FASTA",
)


def _on_upload_change(change):
    uploaded = upload_button.value
    if not uploaded:
        return
    file_info = next(iter(uploaded.values())) if isinstance(uploaded, dict) else uploaded[0]
    content = file_info["content"] if isinstance(file_info, dict) else file_info.content
    fasta_input.value = bytes(content).decode("utf-8", errors="replace")


upload_button.observe(_on_upload_change, names="value")

storage_caption = widgets.HTML(
    "The screening databases are a ~7 GB compressed download that expands to <b>~44 GB uncompressed "
    "on disk</b>. Check this to save them to your <b>Google Drive</b> (make sure you have ~44 GB free "
    "there) so future runs skip re-downloading; leave unchecked to download them just for this "
    "Colab session instead."
)
use_drive_checkbox = widgets.Checkbox(
    value=False,
    description="Save databases to my Google Drive (recommended)",
    indent=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="auto"),
)

run_button = widgets.Button(description="Run", button_style="success", icon="play")
status_html = widgets.HTML(value="")

log_output = widgets.Output(
    layout=widgets.Layout(max_height="320px", overflow_y="auto", border="1px solid #999", padding="4px")
)
log_accordion = widgets.Accordion(children=[log_output])
log_accordion.set_title(0, "Setup / run logs (click to expand)")
log_accordion.selected_index = None  # collapsed by default

report_output = widgets.Output()


def _set_status(message, is_error=False):
    color = "#b00020" if is_error else "inherit"
    status_html.value = f"<b style='color:{color}'>{message}</b>"


def _on_run_clicked(b):
    run_button.disabled = True
    log_output.clear_output()
    report_output.clear_output()
    try:
        run_commec_pipeline(
            job_name_input.value,
            fasta_input.value,
            use_drive_checkbox.value,
            log_output,
            report_output,
            _set_status,
        )
    except Exception as e:
        _set_status(f"ERROR: {e}", is_error=True)
        log_output.append_stdout(f"\nERROR: {e}\n")
        log_accordion.selected_index = 0  # auto-expand logs so the failure is visible
    finally:
        run_button.disabled = False


run_button.on_click(_on_run_clicked)

display(widgets.HTML("<h3>1. Job name</h3>"))
display(job_name_input)
display(widgets.HTML("<h3>2. Provide a FASTA sequence</h3>"))
display(fasta_input)
display(upload_button)
display(widgets.HTML("<h3>3. Database storage</h3>"))
display(storage_caption)
display(use_drive_checkbox)
display(widgets.HTML("<h3>4. Run</h3>"))
display(run_button)
display(status_html)
display(log_accordion)
display(report_output)

if LAST_OUTPUT_DIR and LAST_NAME:
    show_results(LAST_OUTPUT_DIR, LAST_NAME, report_output)